In [5]:
"""Parse Testudo catalog HTML into domain models."""

import re

import sys
from pathlib import Path

project_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(project_root))

from bs4 import BeautifulSoup, Tag
from src.models import Course, Section
from typing import List

class CatalogParser:
    def parse_all_course_prefixes(self, page, tag, class_) -> List[str]:
        soup = BeautifulSoup(page, "html.parser")
        course_prefixes = soup.find_all('span', class_ = 'prefix-abbrev')

        for idx, element in enumerate(course_prefixes):
            course_prefixes[idx] = element.text

        return course_prefixes

    


In [47]:
"""HTTP client for Testudo catalog pages."""

import requests

from src.config import Settings, settings
from typing import List


class CatalogScraper:
    def __init__(self, config: Settings = settings) -> None:
        self.config = config
        self.session = requests.Session()
        self.parser = CatalogParser()
        self.session.headers["User-Agent"] = "UWA data-ingestion/1.0"

    def fetch_course_prefixes(self, semester: str) -> List[str]:
        url = f"{self.config.testudo_base_url}/{semester}"
        response = self.session.get(
            url, timeout=self.config.request_timeout_seconds
        )
        response.raise_for_status()
        course_prefixes = self.parser.parse_all_course_prefixes(response.text, 'span', 'prefix_abbrev')
        return course_prefixes

    def close(self) -> None:
        self.session.close()

    def __enter__(self) -> "CatalogScraper":
        return self

    def __exit__(self, *_: object) -> None:
        self.close()

payload = {
    "courseId": "CMSC",
    "sectionId": "",
    "termId": "202608",

    "_openSectionsOnly": "on",

    "creditCompare": ">=",
    "credits": "0.0",
    "courseLevelFilter": "ALL",
    "instructor": "",

    "_facetoface": "on",
    "_blended": "on",
    "_online": "on",

    "courseStartCompare": "",
    "courseStartHour": "",
    "courseStartMin": "",
    "courseStartAM": "",

    "courseEndHour": "",
    "courseEndMin": "",
    "courseEndAM": "",

    "teachingCenter": "ALL",

    "_classDay1": "on",
    "_classDay2": "on",
    "_classDay3": "on",
    "_classDay4": "on",
    "_classDay5": "on",
}


In [73]:
url = "https://app.testudo.umd.edu/soc/search"

import requests

payload['courseId'] = "MATH"
payload['termID'] = '202608'

response = requests.get(
    url, timeout=30, params = payload
)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

In [74]:
unparsed_courses = soup.find_all("div", class_="course")
unparsed_courses[0]

<div class="course" id="MATH003">
<input name="courseId" type="hidden" value="MATH003"/>
<div class="row">
<div class="course-id-container one columns">
<div class="course-id">MATH003</div>
</div>
<div class="course-info-container eleven columns">
<div class="course-basic-info-container sixteen colgrid">
<div class="row">
<div class="thirteen columns">
<span class="course-title">Developmental Mathematics</span>
</div>
<div class="two columns">
<fieldset class="syllabus-fieldset">
<legend>
<a class="toggle-syllabus-link" href="">
<span class="toggle-sections-arrow ui-icon ui-icon-triangle-1-e"></span>
<span class="toggle-sections-link-text">Syllabus Repository  </span>
<span>(1)</span>
<input name="courseId" type="hidden" value="MATH003"/>
</a>
</legend>
<div id="MATH003-syllabus-container"></div>
</fieldset>
</div>
<!-- @@@@@@@@@ -->
<div class="course-action-links-container one columns">
<a class="saved-course-section-toggle-link" href="javascript:void(0)" onclick="SocJs.saveSection(t

In [77]:
from dataclasses import dataclass, field


@dataclass(frozen=True, slots=True)
class Meeting:
    days: tuple[str, ...]
    start_time: str
    end_time: str
    location: str | None = None


@dataclass(frozen=True, slots=True)
class Section:
    course_code: str
    section_code: str

    instructors: tuple[str, ...] = field(default_factory=tuple)
    meetings: tuple[Meeting, ...] = field(default_factory=tuple)

    open_seats: int | None = None
    total_seats: int | None = None
    waitlist_count: int | None = None


@dataclass(frozen=True, slots=True)
class Course:
    code: str
    semester: str
    title: str

    description: str | None = None
    credits: int | None = None

    sections: tuple[Section, ...] = field(default_factory=tuple)


def _get_text(element, default=None):
    if element is None:
        return default

    return element.get_text(strip=True)


def _get_int(element):
    value = _get_text(element)

    if value is None:
        return None

    try:
        return int(value)
    except ValueError:
        return None


def _parse_days(days_text: str | None) -> tuple[str, ...]:
    if not days_text:
        return tuple()

    # Example:
    # "MWF" -> ("M", "W", "F")
    return tuple(days_text)


def _parse_course(unparsed_course, semester):
    course_code = _get_text(
        unparsed_course.find("div", class_="course-id")
    )

    course_title = _get_text(
        unparsed_course.find("span", class_="course-title"),
        default="",
    )

    course_description = _get_text(
        unparsed_course.find("div", class_="approved-course-text")
    )

    course_credits = _get_int(
        unparsed_course.find("span", class_="course-min-credits")
    )

    all_section_data = unparsed_course.find_all(
        "div",
        class_="section delivery-f2f",
    )

    sections = []

    for section_data in all_section_data:
        section_code = _get_text(
            section_data.find("span", class_="section-id")
        )

        instructor_elements = section_data.find_all(
            "span",
            class_="section-instructor",
        )

        instructors = tuple(
            instructor.get_text(strip=True)
            for instructor in instructor_elements
        )

        total_seats = _get_int(
            section_data.find("span", class_="total-seats-count")
        )

        open_seats = _get_int(
            section_data.find("span", class_="open-seats-count")
        )

        waitlist_count = _get_int(
            section_data.find("span", class_="waitlist-count")
        )

        meetings = []

        class_meetings = section_data.find_all(
            "div",
            class_="class-days-container",
        )

        for meeting_data in class_meetings:
            days_text = _get_text(
                meeting_data.find("span", class_="section-days")
            )

            start_time = _get_text(
                meeting_data.find("span", class_="class-start-time")
            )

            end_time = _get_text(
                meeting_data.find("span", class_="class-end-time")
            )

            location = _get_text(
                meeting_data.find(
                    "div",
                    class_="section-class-building-group",
                )
            )

            meeting = Meeting(
                days=_parse_days(days_text),
                start_time=start_time or "",
                end_time=end_time or "",
                location=location,
            )

            meetings.append(meeting)

        section = Section(
            course_code=course_code,
            section_code=section_code,
            instructors=instructors,
            meetings=tuple(meetings),
            open_seats=open_seats,
            total_seats=total_seats,
            waitlist_count=waitlist_count,
        )

        sections.append(section)

    course = Course(
        code=course_code,
        semester=semester,
        title=course_title,
        description=course_description,
        credits=course_credits,
        sections=tuple(sections),
    )

    return course


def parse_courses(unparsed_courses, semester):
    courses = []

    for unparsed_course in unparsed_courses:
        course = _parse_course(
            unparsed_course,
            semester,
        )

        courses.append(course)

    return courses


courses = parse_courses(
    [unparsed_courses[0]],
    semester="202608",
)

print(courses[0])

Course(code='MATH003', semester='202608', title='Developmental Mathematics', description="A review of Intermediate High School Algebra intended for students preparing for one of the credit bearing Fundamental Studies Math Courses. It is taught in special computer labs using a self-paced computer program. The curriculum will be geared toward the student's level of algebra skills and eventual goals. There is a special fee for the course that may be applied in addition to the regular tuition charge. Students should refer to the schedule of classes for details on fees as they apply to a particular semester. The course does not carry any credit toward any degree at the University. The course is repeatable. Topics will be chosen from exponents, polynomials, linear equations, quadratic equations as well as polynomial, rational, exponential and logarithm functions and elementary probability or statistics, depending on the student.", credits=3, sections=(Section(course_code='MATH003', section_c

In [76]:
response.url

'https://app.testudo.umd.edu/soc/search?courseId=MATH&sectionId=&termId=202608&_openSectionsOnly=on&creditCompare=%3E%3D&credits=0.0&courseLevelFilter=ALL&instructor=&_facetoface=on&_blended=on&_online=on&courseStartCompare=&courseStartHour=&courseStartMin=&courseStartAM=&courseEndHour=&courseEndMin=&courseEndAM=&teachingCenter=ALL&_classDay1=on&_classDay2=on&_classDay3=on&_classDay4=on&_classDay5=on&termID=202608'